In questo notebook si vogliono valutare le performance del sistema.
Per fare ciò viene utilizzato il dataset MemeCap, che contiene immagini e didascalie associate prodotte da esseri umani assunti tramite Amazon Mechanical Turk.
Verrà effettuato il sampling di n=100 meme dal dataset.
Per questi meme verrà calcolata la similarità tra gli embedding delle didascalie generate dal modello e gli embedding di quelle generate dagli esseri umani.
La similarità verrà calcolata tramite la cosine similarity.

In [1]:
import json

data= json.load(open("memecap_dataset\meme-cap\data\memes-test.json"))
print(len(data))

559


In [2]:
print (data[0].keys())  

dict_keys(['category', 'img_captions', 'meme_captions', 'title', 'url', 'img_fname', 'metaphors', 'post_id'])


In [3]:
data = data[:100]

In [4]:
# 1. Calcolo gli embedding
def join_text_fields (meme_entry):
    text=", ".join(meme_entry["img_captions"])+ ", ".join(meme_entry["meme_captions"])
    return text

from meme_analysis_pipeline.components.local_components import ClipEmbedder
clip_embedder = ClipEmbedder()
for meme_entry in data:
    meme_entry["text"]=join_text_fields(meme_entry)
    meme_entry["clip_embedding"]=clip_embedder.calculate_text_embedding(meme_entry["text"])


d:\Desktop\Git\hypermeme\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
data[0]["clip_embedding"].squeeze(0).detach().numpy()

(512,)

In [7]:
from meme_analysis_pipeline.pipeline import MemeAnalysisPipeline
from meme_analysis_pipeline.components.local_components import *
from meme_analysis_pipeline.components.remote_components import GoogleLlmDescriber
import time
pipeline = MemeAnalysisPipeline(describer=GoogleLlmDescriber(), embedding_calculator=ClipEmbedder(), database_manager=ElasticSearchManager(), image_downloader=DummyImgDownloader())
for meme_entry in data:
    meme_entry["enriched_meme"]=pipeline.process_meme(image_url=meme_entry["url"], post_text=meme_entry["title"], local_url="memecap_dataset\memes\\"+meme_entry ["img_fname"])
    time.sleep(3)


In [22]:
#compute the average cosine similarity between the embeddings

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity_matrix = cosine_similarity([meme_entry["clip_embedding"].squeeze(0) for meme_entry in data], [meme_entry["enriched_meme"].text_embedding.squeeze(0) for meme_entry in data])
similarity_matrix

array([[0.5016937 , 0.27602065, 0.2812265 , ..., 0.26477394, 0.2829304 ,
        0.27571207],
       [0.52244794, 0.77809435, 0.37736538, ..., 0.64480287, 0.505735  ,
        0.612162  ],
       [0.4509241 , 0.5365623 , 0.7434678 , ..., 0.5188663 , 0.43764287,
        0.51375276],
       ...,
       [0.446657  , 0.54464257, 0.31121027, ..., 0.7078364 , 0.44298944,
        0.53674746],
       [0.48805442, 0.5231516 , 0.3700164 , ..., 0.5846009 , 0.6643616 ,
        0.52252704],
       [0.49394393, 0.5295728 , 0.263851  , ..., 0.46848884, 0.42650542,
        0.77543473]], shape=(100, 100), dtype=float32)